In [2]:
# Install library yang dibutuhkan
!pip install ultralytics roboflow mlflow opencv-python numpy

In [22]:
import os
from roboflow import Roboflow

# 1. KONFIGURASI API & PROJECT
API_KEY = "dLHzVL3NtrVax10kPOoA" 
WORKSPACE = "roboflow-100"
PROJECT_NAME = "construction-safety-gsnvb"
VERSION = 1

# Inisialisasi koneksi ke Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
version = project.version(VERSION)


# 2. VERIFIKASI KETERSEDIAAN FILE LOKAL
# Secara default, Roboflow mengunduh ke folder dengan format "nama_project-versi"
expected_folder_name = f"{PROJECT_NAME}-{VERSION}"
yaml_file_path = os.path.join(expected_folder_name, "data.yaml")

# Cek apakah file data.yaml dari dataset tersebut sudah ada di direktori saat ini
if os.path.exists(yaml_file_path):
    print(f"\n✅ Dataset sudah tersedia di lokal ('{expected_folder_name}').")

    # Simpan lokasi absolut agar bisa digunakan oleh cell training selanjutnya
    dataset_location = os.path.abspath(expected_folder_name)
    
else:
    print("\n Dataset belum ditemukan di lokal. Memulai proses download...")
    
    # Proses download hanya berjalan jika folder belum ada
    dataset = version.download("yolov11")
    dataset_location = dataset.location

# Variabel dataset_location ini yang akan kita panggil di Cell Training/Augmentasi
print(f"\n✅dataset siap digunakan: {dataset_location}")

loading Roboflow workspace...
loading Roboflow project...
2026/05/13 01:40:08 INFO:     127.0.0.1:35356 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK

 Dataset belum ditemukan di lokal. Memulai proses download...

 dataset siap digunakan: /home/azunya/Documents/kuliah/sem6/matkul/data mining/project/K3_DETECTION_YOLOV11-new/construction-safety-1
2026/05/13 01:40:38 INFO:     127.0.0.1:38752 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/13 01:41:08 INFO:     127.0.0.1:37666 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/13 01:41:11 INFO:     127.0.0.1:37666 - "GET / HTTP/1.1" 304 Not Modified
2026/05/13 01:41:11 INFO:     127.0.0.1:37666 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
2026/05/13 01:41:11 INFO:     127.0.0.1:37666 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
2026/05/13 01:41:11 INFO:     127.0.0.1:37698 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
2026/05/13 01:41:11 INFO:     127.0

t=2026-05-13T01:46:24+0700 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=48a31fcfe921 clientid=588e3d310f18e7c4439a521b4f86ef5c
t=2026-05-13T01:46:24+0700 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=57a93eb75751 err="session closed"


2026/05/13 01:46:24 INFO:     127.0.0.1:33200 - "GET /ajax-api/2.0/mlflow/model-versions/search?filter=tags.%60mlflow.prompt.is_prompt%60+%3D+%27true%27+AND+tags.%60mlflow.prompt.run_ids%60+ILIKE+%22%25a7f7d687f1fd40b999359efd5702e528%25%22 HTTP/1.1" 200 OK
2026/05/13 01:46:24 INFO:     127.0.0.1:33214 - "GET /ajax-api/2.0/mlflow/model-versions/search?filter=run_id%3D%27a7f7d687f1fd40b999359efd5702e528%27 HTTP/1.1" 200 OK
2026/05/13 01:46:24 INFO:     127.0.0.1:33186 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
2026/05/13 01:46:38 INFO:     127.0.0.1:47208 - "POST /ajax-api/3.0/mlflow/traces/search HTTP/1.1" 200 OK
2026/05/13 01:46:38 INFO:     127.0.0.1:47194 - "POST /ajax-api/3.0/mlflow/traces/search HTTP/1.1" 200 OK
2026/05/13 01:46:38 INFO:     127.0.0.1:47228 - "POST /ajax-api/3.0/mlflow/traces/metrics HTTP/1.1" 200 OK
2026/05/13 01:46:38 INFO:     127.0.0.1:47214 - "POST /ajax-api/3.0/mlflow/traces/metrics HTTP/1.1" 200 OK
2026/05/13 01:46:38 INFO:     127.0.0.1:4720

t=2026-05-13T01:58:30+0700 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=4f125d0bd030 clientid=588e3d310f18e7c4439a521b4f86ef5c
t=2026-05-13T01:58:30+0700 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=57a93eb75751 err="session closed"


In [2]:
import os
import cv2
import albumentations as A
import glob
from tqdm import tqdm

# Tentukan path dari dataset yang baru diunduh
train_images_dir = os.path.join(dataset.location, "train", "images")
train_labels_dir = os.path.join(dataset.location, "train", "labels")

# Dapatkan semua file gambar asli
image_files = glob.glob(os.path.join(train_images_dir, "*.jpg"))
print(f"Jumlah gambar asli sebelum augmentasi: {len(image_files)}")

# Target adalah menambah sekitar 3 gambar augmentasi per gambar asli
# 1206 (asli) + (1206 * 2.5 rata-rata) = ~4200 gambar
AUGMENTATIONS_PER_IMAGE = 3 

# Definisikan teknik augmentasi
# Albumentations secara otomatis menyesuaikan Bounding Box YOLO
transform = A.Compose([
    A.HorizontalFlip(p=0.5),                            # Membalik gambar kiri-kanan
    A.RandomBrightnessContrast(p=0.4),                  # Ubah kecerahan/kontras
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.6), # Geser, skala, putar ringan
    A.Blur(blur_limit=3, p=0.2),                        # Efek kamera tidak fokus
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_area=1024, min_visibility=0.1))

count_generated = 0

print("Memulai proses augmentasi data...")
for img_path in tqdm(image_files):
    # Dapatkan nama file tanpa ekstensi
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(train_labels_dir, f"{base_name}.txt")
    
    # Hanya proses jika file labelnya ada
    if not os.path.exists(label_path):
        continue
        
    # Baca gambar
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Albumentations pakai RGB
    
    # Baca label YOLO
    bboxes = []
    class_labels = []
    with open(label_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                # Format YOLO: class x_center y_center width height
                class_id = int(parts[0])
                bbox = [float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])]
                bboxes.append(bbox)
                class_labels.append(class_id)
                
    # Buat variasi gambar baru
    for i in range(AUGMENTATIONS_PER_IMAGE):
        try:
            # Lakukan transformasi
            augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            aug_img = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_labels = augmented['class_labels']
            
            # Abaikan jika augmentasi memotong box terlalu ekstrem sampai hilang
            if len(aug_bboxes) == 0 and len(bboxes) > 0:
                continue
                
            # Konversi kembali ke BGR untuk disimpan dengan cv2
            aug_img_bgr = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            
            # Buat nama file baru
            new_base_name = f"{base_name}_aug_{i}"
            new_img_path = os.path.join(train_images_dir, f"{new_base_name}.jpg")
            new_label_path = os.path.join(train_labels_dir, f"{new_base_name}.txt")
            
            # Simpan gambar baru
            cv2.imwrite(new_img_path, aug_img_bgr)
            
            # Simpan label YOLO baru
            with open(new_label_path, 'w') as f:
                for label, box in zip(aug_labels, aug_bboxes):
                    f.write(f"{label} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")
                    
            count_generated += 1
            
        except Exception as e:
            # Lewati gambar jika ada box yang keluar dari batas perhitungan matematis
            pass

total_images_now = len(glob.glob(os.path.join(train_images_dir, "*.jpg")))
print(f"\nSelesai! Berhasil membuat {count_generated} gambar baru.")
print(f"Total gambar di folder dataset training saat ini: {total_images_now}")

ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.


Jumlah gambar asli sebelum augmentasi: 997
Memulai proses augmentasi data...


100%|████████████████████████████████████████| 997/997 [00:08<00:00, 122.79it/s]


Selesai! Berhasil membuat 2991 gambar baru.
Total gambar di folder dataset training saat ini: 3988


In [3]:
import torch
from ultralytics import YOLO
import os

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("K3_PPE_Detection")

# Memastikan penggunaan GPU
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Menggunakan device: {device}")

# Inisialisasi model
model = YOLO('yolo11n.pt') 
dataset_yaml = os.path.join(dataset.location, "data.yaml")

results = model.train(
    data=dataset_yaml,
    optimizer='AdamW',
    cos_lr=True,
    lr0=0.001,
    patience=20,
    mosaic=0.5,         
    mixup=0.0,          
    degrees=0.0,        
    freeze=10,          
    dropout=0.15,       
    amp=True,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='K3_DETECTION',
    name='yolov11n_roboflow100',
    workers=2,
    cache=False         
)

best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
print(f"Model terbaik disimpan di: {best_model_path}")

Menggunakan device: 0
Ultralytics 8.4.48 🚀 Python-3.14.4 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5773MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/azunya/Documents/kuliah/sem6/matkul/data mining/project/K3_DETECTION_YOLOV11-new/construction-safety-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.15, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosai

The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.


       1/50      1.75G       1.51      1.601      1.566         15        640: 100% ━━━━━━━━━━━━ 250/250 9.3it/s 26.8s<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.4it/s 1.7s0.2s
                   all        119        715      0.779      0.605      0.632      0.297

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.05G       1.44      1.143      1.483         27        640: 100% ━━━━━━━━━━━━ 250/250 11.9it/s 21.1s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 8.5it/s 0.5s0.2s
                   all        119        715      0.738      0.622      0.685      0.358

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.05G       1.41      1.043      1.462         24        640: 100% ━━━━━━━━━━━━ 250/250 11.9it/s 21.0s<0.1s
                 Class     Imag

In [4]:
import subprocess
import time
from pyngrok import ngrok, conf
import os
import mlflow

# 0. AKTIFKAN SYSTEM METRICS
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

# 1. KONFIGURASI DATABASE & NGROK
MLFLOW_DB = "sqlite:///mlflow.db"
NGROK_AUTH_TOKEN = "3DdBpUEORvSaoQWXzTMehc7JGhY_7kAvnMWYZ2vDEeeHTM6FN"

mlflow.set_tracking_uri(MLFLOW_DB)
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
pyngrok_config = conf.PyngrokConfig(region="ap")

# 2. JALANKAN MLFLOW UI & TUNNELING
print("Sedang menjalankan MLflow UI di background...")
proc = subprocess.Popen([
    "mlflow", "server", 
    "--backend-store-uri", MLFLOW_DB, 
    "--port", "5000"
])

time.sleep(5)

try:
    public_url = ngrok.connect(5000, pyngrok_config=pyngrok_config)
    print("\n" + "="*50)
    print(f"🚀 MLflow UI Berhasil di-tunnel!")
    print(f"🔗 Akses link publik kamu di: {public_url}")
    print("="*50 + "\n")
except Exception as e:
    print(f"Gagal membuat tunnel Ngrok: {e}")

# 3. UPLOAD MODEL KE DATABASE (ARTIFACTS)
mlflow.set_experiment("K3_PPE_Detection")
best_model_path = "/home/azunya/Documents/kuliah/sem6/matkul/data mining/project/K3_DETECTION_YOLOV11-new/runs/detect/K3_DETECTION/yolov11n_run-5/weights/best.pt"

with mlflow.start_run(run_name="yolov11n_final_model") as run:
    print("Sedang mengunggah model ke MLflow...")
    
    if os.path.exists(best_model_path):
        # Menyimpan file best.pt ke tab Artifacts
        mlflow.log_artifact(best_model_path, artifact_path="model_terbaik")
        
        # Mencatat metrik
        mlflow.log_metric("mAP50", 0.456)
        mlflow.log_metric("Recall", 0.763)
        mlflow.log_metric("Precision", 0.496)
        
        print("✅ Berhasil! Model 'best.pt' dan metrik telah dicatat di Artifacts.")
    else:
        print("❌ Gagal: File model tidak ditemukan di path tersebut.")

print("Silakan buka/refresh link Ngrok MLflow Anda di browser.")

Sedang menjalankan MLflow UI di background...
Registry store URI not provided. Using backend store URI.


[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/13 07:58:06 ERROR:    [Errno 98] Address already in use
2026/05/13 07:58:11 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/05/13 07:58:11 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/05/13 07:58:11 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/05/13 07:58:11 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



🚀 MLflow UI Berhasil di-tunnel!
🔗 Akses link publik kamu di: NgrokTunnel: "https://brethren-backspace-browse.ngrok-free.dev" -> "http://localhost:5000"

Sedang mengunggah model ke MLflow...
✅ Berhasil! Model 'best.pt' dan metrik telah dicatat di Artifacts.
Silakan buka/refresh link Ngrok MLflow Anda di browser.
2026/05/13 07:58:15 INFO:     127.0.0.1:47494 - "GET / HTTP/1.1" 304 Not Modified
2026/05/13 07:58:16 INFO:     127.0.0.1:47494 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
2026/05/13 07:58:16 INFO:     127.0.0.1:47502 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
2026/05/13 07:58:16 INFO:     127.0.0.1:47494 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
2026/05/13 07:58:16 INFO:     127.0.0.1:47500 - "POST /graphql HTTP/1.1" 200 OK
2026/05/13 07:58:16 INFO:     127.0.0.1:47502 - "GET /ajax-api/2.0/mlflow/experiments/get?experiment_id=1 HTTP/1.1" 200 OK
2026/05/13 07:58:16 INFO:     127.0.0.1:47518 - "POST /ajax-api/2.0/mlflow/experiments

In [1]:
%matplotlib inline 

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. KONFIGURASI MODEL
best_model_path = '/home/azunya/Documents/kuliah/sem6/matkul/data mining/project/K3_DETECTION_YOLOV11-new/runs/detect/K3_DETECTION/yolov11n_roboflow100/weights/best.pt'
model = YOLO(best_model_path)

# 2. PEMBUATAN WIDGET UPLOADER
uploader = widgets.FileUpload(
    accept='image/*',  # Hanya menerima file gambar
    multiple=False     # Hanya menerima 1 file dalam satu waktu
)
out = widgets.Output() # Area untuk menampilkan hasil

# 3. FUNGSI DETEKSI (Berjalan otomatis saat gambar diupload)
def on_upload(change):
    with out:
        # Bersihkan hasil sebelumnya jika kita mengupload gambar baru
        clear_output(wait=True)
        
        if not uploader.value:
            return
            
        try:
            # Mengambil data gambar dari uploader (Mendukung ipywidgets v7 dan v8)
            if isinstance(uploader.value, dict): # Untuk ipywidgets versi lama
                filename = list(uploader.value.keys())[0]
                content = uploader.value[filename]['content']
            else: # Untuk ipywidgets versi baru
                content = uploader.value[0]['content']
                filename = uploader.value[0]['name']
                
            # Konversi file byte (langsung dari memori) ke format gambar OpenCV
            nparr = np.frombuffer(content, np.uint8)
            image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            
            print(f"Mendeteksi gambar: {filename}...")
            
            # --- PROSES YOLO ---
            results = model.predict(source=image, conf=0.25)
            result = results[0]
            annotated_image = result.plot()
            
            # --- CETAK TEKS ---
            print("\n" + "="*50)
            print("="*50)

            if len(result.boxes) == 0:
                print("Tidak ada objek pelanggaran K3 yang terdeteksi.")
            else:
                for i, box in enumerate(result.boxes):
                    class_id = int(box.cls[0])
                    class_name = model.names[class_id]
                    confidence = float(box.conf[0])
                    coords = box.xyxy[0].tolist()
                    x1, y1, x2, y2 = [int(c) for c in coords]
                    
                    print(f"Objek {i+1}:")
                    print(f"  - Kategori : {class_name}")
                    print(f"  - Akurasi  : {confidence * 100:.2f}%")
                    print(f"  - Posisi   : Kotak dari (X:{x1}, Y:{y1}) ke (X:{x2}, Y:{y2})")
                    print("-" * 30)

            # --- TAMPILKAN GAMBAR ---
            annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
            fig = plt.figure(figsize=(12, 12)) 
            plt.imshow(annotated_image_rgb)
            plt.axis('off') 
            plt.title(f"Hasil Deteksi K3 - YOLOv11n\n(Total terdeteksi: {len(result.boxes)} objek)", fontsize=14)
            plt.tight_layout()
            plt.show()
            
            if isinstance(uploader.value, tuple):
                uploader.value = tuple()
            else:
                uploader.value.clear()
            uploader._counter = 0 
            
        except Exception as e:
            print(f"Terjadi kesalahan saat memproses gambar: {e}")

# Hubungkan tombol uploader dengan fungsi on_upload
uploader.observe(on_upload, names='value')

# 4. TAMPILKAN ANTARMUKA
display(widgets.VBox([widgets.Label(value="Upload Gambar"), uploader, out]))